In [ ]:
# ACTIVATE FIRST!
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(OPENAI_API_KEY))

First, let's define a function to get a narrative response from OpenAI based on a user's prompt. You can adjust the `model` and `temperature` parameters to get different styles of output.

In [ ]:
def get_narrative_response(user_query, conversation_history):
    try:
        # Append the new user message to the conversation history
        conversation_history.append({"role": "user", "content": user_query})

        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=conversation_history, # Use the dynamic conversation history
            temperature=0.1,
            max_tokens=150
        )
        ai_response_content = completion.choices[0].message.content
        # Append the AI's response to the conversation history
        conversation_history.append({"role": "assistant", "content": ai_response_content})

        return ai_response_content, conversation_history # Return both narrative and updated history
    except Exception as e:
        return f"An error occurred: {e}", conversation_history # Return error and history

Next, we'll create a function to simulate a typewriter effect when displaying the text. This will print the text character by character with a small delay between each. This is a visual effect that runs in the notebook output.

In [ ]:
import time
from IPython.display import display, HTML, clear_output

def typewriter_display(text, delay=0.05):
    html_output = ""
    for char in text:
        html_output += char
        clear_output(wait=True)
        display(HTML(f"<pre style='font-family: monospace; white-space: pre-wrap;'>{html_output}</pre>"))
        time.sleep(delay)

# Example usage:
# typewriter_display("Hello, this is a typewriter effect!")

Now, let's put it all together. You can enter your query, and the AI will generate a narrative that will then be displayed with the typewriter effect.

In [ ]:
# @title Enter your query and run this cell to get a narrative response with typewriter effect
user_query = "I dreamt I was flying." # @param {type:"string"}

# Initialize conversation history for this single query run (if still desired)
# However, for a continuous ELIZA-style system, a new loop will be more appropriate.
conversation_history = [
    {"role": "system", "content": "You are a whimsical storyteller, crafting narrative descriptions from user inputs about their day or dreams. Respond with creative, descriptive language."}
]

narrative_response, conversation_history = get_narrative_response(user_query, conversation_history)

if narrative_response:
    print("\nAI's narrative:")
    typewriter_display(narrative_response)
else:
    print("Could not generate a narrative.")

### Start the ELIZA-style conversation

Run this cell to start an interactive conversation. Type your queries, and the AI will respond. Type `exit` to end the conversation.

In [ ]:
conversation_history = [
    {"role": "system", "content": "You are a whimsical storyteller, crafting narrative descriptions from user inputs about their day or dreams. Respond with creative, descriptive language."}
]

print("Welcome to the interactive storyteller! Type 'exit' to end the conversation.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Ending conversation. Goodbye!")
        break

    narrative_response, conversation_history = get_narrative_response(user_input, conversation_history)

    if narrative_response:
        print("\nAI's narrative:")
        typewriter_display(narrative_response)
    else:
        print("Could not generate a narrative.")
    print("\n") # Add a newline for better readability between turns



You: exit
Ending conversation. Goodbye!


First, let's define a function to generate an image using OpenAI's DALL-E API. This function will take a text prompt and return the URL of the generated image.

In [ ]:
from IPython.display import Image, display

def generate_image_from_prompt(prompt):
    try:
        # Generate image using DALL-E 2
        response = client.images.generate(
            model="dall-e-2",
            prompt=prompt,
            size="256x256", # Reverted to 256x256, which is a valid option for dall-e-2
            n=1,
        )
        image_url = response.data[0].url
        return image_url
    except Exception as e:
        return f"An error occurred during image generation: {e}"

In [ ]:
def get_narrative_response(user_query, conversation_history, image_url=None):
    try:
        user_message_content = []
        if user_query:
            user_message_content.append({"type": "text", "text": user_query})
        if image_url:
            user_message_content.append({"type": "image_url", "image_url": {"url": image_url}})

        # If there's no user query or image, we should not add an empty message
        if not user_message_content:
            return "Please provide a query or an image.", conversation_history

        # Append the new user message (with optional image) to the conversation history
        conversation_history.append({"role": "user", "content": user_message_content})

        completion = client.chat.completions.create(
            model="gpt-4o-mini", # Changed model to gpt-4o-mini as requested
            messages=conversation_history, # Use the dynamic conversation history
            temperature=0.1,
            max_tokens=150
        )
        ai_response_content = completion.choices[0].message.content
        # Append the AI's response to the conversation history
        conversation_history.append({"role": "assistant", "content": ai_response_content})

        return ai_response_content, conversation_history # Return both narrative and updated history
    except Exception as e:
        return f"An error occurred: {e}", conversation_history # Return error and history

Now, let's modify the ELIZA-style conversation loop to include image generation and display.

In [ ]:
conversation_history = [
    {"role": "system", "content": "You are a whimsical storyteller, crafting narrative descriptions from user inputs about their day or dreams. Respond with creative, descriptive language."}
]

print("Welcome to the interactive storyteller! Type 'exit' to end the conversation.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Ending conversation. Goodbye!")
        break

    # Prompt for an image URL
    image_input = input("Enter an image URL (optional, press Enter to skip): ")
    image_url_for_narrative = image_input if image_input.strip() else None

    # If the user provides an image URL for the narrative, we might not want to generate another one
    # Or we can generate based on text input if no image is provided, or even generate based on the image's content
    # For now, let's keep the existing image generation based on user_input for DALL-E

    narrative_response, conversation_history = get_narrative_response(user_input, conversation_history, image_url=image_url_for_narrative)

    if narrative_response:
        print("\nAI's narrative:")
        typewriter_display(narrative_response)

        # Generate and display image based on user input (for DALL-E image generation)
        # This part remains separate from the image input for the narrative
        print("Generating an image from your text input...")
        image_url_generated = generate_image_from_prompt(user_input)
        if image_url_generated and not image_url_generated.startswith("An error occurred"):
            display(Image(url=image_url_generated))
            print(f"Generated Image URL: {image_url_generated}") # Optional: print URL
        else:
            print(f"Could not generate image: {image_url_generated}")
    else:
        print("Could not generate a narrative.")
    print("\n") # Add a newline for better readability between turns

Welcome to the interactive storyteller! Type 'exit' to end the conversation.



KeyboardInterrupt: Interrupted by user